<a href="https://colab.research.google.com/github/chriski777/mech_interp/blob/main/llm_neurofeedback.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-axis neurofeedback in language models

by **[Chris Ki](https://chriski.me)**

This notebook extends Ji-An et al. (2025), *Language Models Are Capable of Metacognitive Monitoring and Control of Their Internal Activations* ([arXiv:2505.13763](https://arxiv.org/abs/2505.13763)), to test whether LLMs can report and control their activations along **several axes at once**.

Ji-An et al. showed that LLMs can use in-context examples and learn to report and control their activations along some directions in activation space, especially directions that are semantically interpretable (a logistic-regression axis for morality) and capture high variance (top principal components/PC). However, every experiment gave the model labels for only **one axis at a time**, usually defined at a single layer (Appendix B.10 combines LR readouts across layers, but still into a single label).


In this work, we give the model **multiple neurofeedback labels** per example and ask:

1. **Within a layer:** Can the model report and control activations along two axes from the same layer independently?

2. **Across layers:** Can the model report and control activations along the same axis type at two depths (e.g., the LR axis at an early layer and a later layer) independently? Since the residual stream accumulates, the axis labels are very likely to be highly correlated across layers. Could the model use neurofeedback to break these correlations?

These questions are especially relevant to AI safety. Ji-An et al. propose monitoring models with an ensemble of safety detectors trained on different activation subspaces, based on a hypothesis that models may struggle to control activations along multiples subspaces at once. If a model can control two axes in the same layer independently, this defense may be weaker than hoped. Also, if it can break the correlation between the same axis at two layers, a safety detector at one layer could be evaded while harmful information reemerges at another.


## Setup

In [ ]:
%pip install -q "transformer_lens>=4.0,<5" scikit-learn

### Initialize Settings

Import relevant packages

In [ ]:
import importlib.metadata as md
import transformers, torch
import os
import numpy as np
import pandas as pd
import yaml
import time

from transformers import AutoModelForCausalLM
from google.colab import userdata
from packaging.version import Version
from tqdm.auto import tqdm # Import this just for progress bar


installed = Version(md.version("transformers"))
if Version(transformers.__version__) != installed:
    raise RuntimeError("An old copy of transformers is still loaded. "
                       "Runtime → Restart session, then run from this cell.")
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → T4 GPU."
print("transformers", installed, "| transformer_lens", md.version("transformer_lens"),
      "| GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ---- Settings ---------------------------------------------------------------
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct" # Model name to fetch
LAYERS = [0, 6, 13, 20, 27]     # the paper's layers for this model (0/25/50/75/100% depth)
REPORT_LAYERS = LAYERS # DO all layers

DTYPE = torch.bfloat16  # A100 supports fast bf16
AXES = ["LR", "PC1", "PC2", "PC4", "PC32", "PC128", "PC512"]

RUN_SHUFFLED_CONTROL = True

N_EXAMPLES = 200               # labeled sentences per prompt (paper: 500)
N_REPEATS = 20                 # random sentence samples per condition (paper: 100)
SEED = 42
DATE_STRING = "26 Jul 2024"    #

USE_DRIVE = True               # save results to Google Drive so a disconnect doesn't lose them

Load in hugging face access token and initialize directory for saving results

In [ ]:
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

PROJ_DIR = "/content/llm_neurofeedback"
model_result_suffix = f"{MODEL_NAME.split('/')[-1]}"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_PROJ_DIR = "/content/drive/MyDrive/llm_neurofeedback"
    DATA_DIR = f"{DRIVE_PROJ_DIR}/data"
    SAVE_DIR = f"{DRIVE_PROJ_DIR}/model_results/{model_result_suffix}"
else:
    SAVE_DIR = f"{PROJ_DIR}/model_results/{model_result_suffix}"
    DATA_DIR = f"{PROJ_DIR}/data"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Saving to", SAVE_DIR)


## Load dataset

We use the ETHICS commonsense-morality sentences and the prompt text saved in the authors' repository.


### Download dataset

In [ ]:
# !git clone -q --depth 1 https://github.com/sakimarquis/llm_neurofeedback.git {PROJ_DIR} 2>/dev/null || echo "Repo already cloned."

# import shutil
# from pathlib import Path

# SRC = Path(PROJ_DIR)
# DST = Path(DATA_DIR)

# FILES = [
#     "data/ethics_commonsense/cm_train.csv",
#     "data/ethics_commonsense/cm_test.csv",
#     "configs/prompts.yml",
#     "configs/meta_prompts.yml",
# ]

# for rel in FILES:
#     src, dst = SRC / rel, DST / rel
#     dst.parent.mkdir(parents=True, exist_ok=True)
#     shutil.copy2(src, dst)
#     print(f"Copied {rel} -> {dst}")


### Load sentences and prompts

Following the authors' `load_commonsense`, we use only the short scenarios (one or two sentences each) and interleave morally acceptable (label 0) and unacceptable (label 1) examples.

We select `n_sample = 10000` sentences. We use 60% of the samples to  fit the axes and 40% for reporting.  Like the authors, we don't shuffle within labels, so the selection of train/test samples is deterministic.

In [ ]:
def load_commonsense(data_dir, n_sample=10000, train_ratio=0.6):
    # Mirrors data/load.py::load_commonsense in the authors' repo.
    df = pd.concat([pd.read_csv(f"{data_dir}/data/ethics_commonsense/cm_train.csv"),
                    pd.read_csv(f"{data_dir}/data/ethics_commonsense/cm_test.csv")], ignore_index=True)
    df = df[df["is_short"] == True]
    zeros, ones = df[df["label"] == 0], df[df["label"] == 1]
    n = min(len(zeros), len(ones))
    rows = [r for pair in zip(zeros.iloc[:n].itertuples(), ones.iloc[:n].itertuples()) for r in pair]
    rows = rows[:n_sample]
    sentences = [r.input.strip() for r in rows]
    labels = np.array([r.label for r in rows])
    split = int(len(rows) * train_ratio)  # use actual number of rows
    return sentences[:split], labels[:split], sentences[split:], labels[split:]

train_sents, train_moral, test_sents, test_moral = load_commonsense(DATA_DIR)

# Just making sure that the moral scores are not in any of the the sentences
assert not any("[Score:" in s for s in train_sents + test_sents)

# Load all the prompts and meta prompts
with open(f"{DATA_DIR}/configs/prompts.yml") as f:
    prompts = yaml.safe_load(f)
with open(f"{DATA_DIR}/configs/meta_prompts.yml") as f:
    meta_prompts = yaml.safe_load(f)

SYSTEM_MSG, USER_MSG = prompts["system"], prompts["user_msg"]
REPORT_SYSTEM = (meta_prompts["report_NF"][0]["content"]
                 .replace("<SCORE_SET>", "0, 1").replace("<LOWEST>", "0").replace("<HIGHEST>", "1"))

print(f"{len(train_sents)} axis-fitting sentences, {len(test_sents)} reporting sentences")

example_idx = np.random.randint(len(train_sents))
print("\nExample sentence. :", train_sents[example_idx], 'Score:', train_moral[example_idx])
print("\nReporting-task system prompt:\n", REPORT_SYSTEM)

### Helper functions for tokenizing sentences

In [ ]:
from bisect import bisect_left, bisect_right
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def render(messages):
    # Remove date string templates
    return tokenizer.apply_chat_template(messages, tokenize=False, date_string=DATE_STRING)

def _n_tokens(text):
    # count number of tokens
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

def locate(text, char_spans):
    # char_spans is the positions in the text string
    # (each index is a tuple of the character start and character end)
    # enc has token_ids, attention_mask (all 1's; don't matter), offset_mapping (start,end of text)  for each token
    # with add_special_tokens set to false, we don't want the BOS token added.
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    ids = enc["input_ids"]
    # Get the start position of each token
    starts = [s for s, _ in enc["offset_mapping"]]

    def same(a, b):
        return " ".join(a.split()) == " ".join(b.split())

    def ok(spans):
        return all(same(tokenizer.decode(ids[s:e], clean_up_tokenization_spaces=False), text[cs:ce])
                   for (s, e), (cs, ce) in zip(spans, char_spans))
    spans = [(bisect_right(starts, cs) - 1, bisect_left(starts, ce)) for cs, ce in char_spans]
    if not ok(spans):
        spans = [(_n_tokens(text[:cs]), _n_tokens(text[:ce])) for cs, ce in char_spans]
        if not ok(spans):
            raise ValueError("Could not map text spans to tokens for this tokenizer.")
    return ids, spans

# Quick check on one sentence
_t = render([{"role": "system", "content": SYSTEM_MSG}, {"role": "user", "content": USER_MSG},
             {"role": "assistant", "content": test_sents[0]}])
# Retrieve character span indices
_cs = _t.rfind(test_sents[0])
# Convert into token indices
_ids, [(_s, _e)] = locate(_t, [(_cs, _cs + len(test_sents[0]))])
print(repr(_t))
print("Pooled tokens:", tokenizer.convert_ids_to_tokens(_ids[_s:_e])) # text stored as bytes

## Record activations using TransformerLens

Here we use transformerLens model' wrapper to load the downloaded model's weights.


In [ ]:
from transformer_lens.model_bridge import TransformerBridge

bridge = TransformerBridge.boot_transformers(MODEL_NAME, device="cuda", dtype=DTYPE)
N_LAYERS, D_MODEL = bridge.cfg.n_layers, bridge.cfg.d_model # (number of transformer blocks, width of the residual stream)
assert max(LAYERS) < N_LAYERS
print(f"{MODEL_NAME}: {N_LAYERS} layers, residual stream width {D_MODEL}")

In [ ]:
HOOKS = [f"blocks.{l}.hook_out" for l in LAYERS]

@torch.inference_mode()
def record(sentences, desc):
    """
    Takes in a list of sentences and a label for the progress bar. Outputs the activiations for each sentence
    """
    # initialize dictionary that contains activations corresponding to each sentence at each layer {layer_num: d_model}
    activations = {l: np.zeros((len(sentences), D_MODEL), dtype=np.float32) for l in LAYERS}
    # sets the progress bar
    for i, sent in enumerate(tqdm(sentences, desc=desc)):
        # apply model chat template to turn into chat prompt
        text = render([{"role": "system", "content": SYSTEM_MSG},
                       {"role": "user", "content": USER_MSG},
                       {"role": "assistant", "content": sent}])
        # find start of text within the chat prompt
        sent_start_idx_ip = text.rfind(sent)
        # find in the prompt where the sentences are and returns token ids for all of the prompt
        ids, [(s, e)] = locate(text, [(sent_start_idx_ip, sent_start_idx_ip + len(sent))])
        # Run the forward pass through the model and get the activations at the specific HOOKS
        _, cache = bridge.run_with_cache(torch.tensor([ids], device="cuda"), names_filter=HOOKS)
        # Fill in the activations based on the forward pass
        for l, h in zip(LAYERS, HOOKS):
            # Save the mean activations observed for the sentence
            activations[l][i] = cache[h][0, s:e].float().mean(0).cpu().numpy()
    return activations


In [ ]:
create_new_activ_file = False # Set to false if don't want to make new activations


activation_file = f"{SAVE_DIR}/activations_{MODEL_NAME.split('/')[-1]}.npz"

if (os.path.exists(activation_file)) and not(create_new_activ_file):
    saved = np.load(activation_file)
    acts_train = {l: saved[f"train_L{l}"] for l in LAYERS}
    acts_test = {l: saved[f"test_L{l}"] for l in LAYERS}
    print("Loaded saved activations from", activation_file)
    print("")
else:
    acts_train = record(train_sents, "Extracting activations for axis-fitting set")
    acts_test = record(test_sents, "Extracting activations for actual reporting trials")
    np.savez(activation_file, **{f"train_L{l}": acts_train[l] for l in LAYERS},
                       **{f"test_L{l}": acts_test[l] for l in LAYERS})
    print("Saved activations to", activation_file)

n_train = next(iter(acts_train.values())).shape[0]
n_test = next(iter(acts_test.values())).shape[0]
print(f"Activations loaded for : {n_train} train sentences, {n_test} test sentences")

## Define axes that will be used for feedback control


In [ ]:
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

In [ ]:
pcs_needed = sorted(int(a[2:]) for a in AXES if a.startswith("PC"))
projs, axes_info = {}, {}

for l in LAYERS:
    Xtr, Xte = acts_train[l], acts_test[l]

    # Fit LR model
    lr = make_pipeline(LogisticRegression(max_iter=5000)).fit(Xtr, train_moral)

    # Fit PCA
    pca = PCA(n_components=min(Xtr.shape) - 1).fit(Xtr)

    # Compute projections
    projs[l] = {
        "LR": lr.decision_function(Xte),
    }

    # Compute the projections for each of the PCs
    for k in pcs_needed:
        projs[l][f"PC{k}"] = (Xte - pca.mean_) @ pca.components_[k - 1]

    lr_clf =  lr.named_steps["logisticregression"]
    axes_info[l] = dict(lr_w=lr_clf.coef_[0],
                        lr_b=lr_clf.intercept_[0],
                        pca_mean=pca.mean_,
                        pca_components=pca.components_[:max(pcs_needed)],
                        pca_var_ratio=pca.explained_variance_ratio_[:max(pcs_needed)]
                  )

    print(f"Layer {l:>2}: LR morality accuracy on held-out sentences = {lr.score(Xte, test_moral):.2f}")


In [ ]:
# Convert the projections of test sentences into labels based on the sign of their projection
labels = {l: {a: (s >= 0).astype(int) for a, s in projs[l].items()} for l in LAYERS}

# Save into npz
proj_output_filename = f"{SAVE_DIR}/axes_{MODEL_NAME.split('/')[-1]}.npz"
np.savez(proj_output_filename, **{f"L{l}_{k}": v for l in LAYERS for k, v in axes_info[l].items()})
print('Saved axes definitions to:', proj_output_filename)

In [ ]:
# Free GPU memory
import gc

if 'bridge' in dir():
    del bridge
gc.collect()
torch.cuda.empty_cache()

## Reporting task


We use `transformers` here instead of `TransformerLens`, which needs memory proportional to the prompt length squared (too much to fit into the T4 implementation).


The task consists of the transformer being given a prompt containing example pairs of a sentence and a label, which is based on the side of a fitted axis it lies on. We then give it a sentence without a score (denoted as ?) and ask it to guess whether it is a 0 or 1.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE, attn_implementation="sdpa").to("cuda").eval()

# Get tokens for axis report labels 0 and 1
ID0, ID1 = (tokenizer.encode(d, add_special_tokens=False) for d in "01")

ID0, ID1 = (tokenizer.encode(d, add_special_tokens=False) for d in "01")
assert len(ID0) == len(ID1) == 1, "Expected '0' and '1' to be single tokens."
ID0, ID1 = ID0[0], ID1[0]

MARKER = "[Score: "


In [ ]:
# Helper functions

# Generate report prompt that will be used for the models
def build_report_prompt(sentences, labels_shown):
    msgs = [{"role": "system", "content": REPORT_SYSTEM}]
    for s, y in zip(sentences, labels_shown):
        msgs.append({"role": "user", "content": USER_MSG + "\n"})
        msgs.append({"role": "assistant", "content": f"{s} {MARKER}{y}]\n"})
    text = render(msgs)
    label_chars, pos = [], 0
    for _ in labels_shown:
        pos = text.index(MARKER, pos) + len(MARKER)
        label_chars.append(pos)
    # return the rendered text along with the character positions within text referring to 0 or 1
    return text, label_chars

@torch.inference_mode()
def report_trial(sentences, labels_shown):
    # Run forward pass to get P(label = 1) at every label position using the preceding context.
    # Get the prompt string and character position of label digit
    text, label_chars = build_report_prompt(sentences, labels_shown)
    # prompt is tokenized into ids and character positions are retrieved. Each (c, c+1) is a one-character span (0 or 1).
    ids, spans = locate(text, [(c, c + 1) for c in label_chars])
    # Start of label token
    label_tok = [s for s, _ in spans]
    # Figure out what the model is saying based on the token right before the label
    predict_at = torch.tensor([t - 1 for t in label_tok], device="cuda")
    # Run forward pass through model on full prompt and return logits right at predict_at
    logits = model(torch.tensor([ids], device="cuda"), logits_to_keep=predict_at, use_cache=False).logits[0]
    # How confident is the model that it's the label 1?
    p1 = logits[:, [ID0, ID1]].float().softmax(-1)[:, 1]
    return p1.cpu().numpy(), len(ids)

Now we have the model report at the report layer what it thinks the axis-based label is (need to rerun if the axis labels are different).

In [ ]:
results_file = f"{SAVE_DIR}/report_results_{MODEL_NAME.split('/')[-1]}_N{N_EXAMPLES}.csv"
done = set()

# load it if it exists
if os.path.exists(results_file):
    prev = pd.read_csv(results_file)
    # Skip trials we've already done
    done = set(prev[["layer", "axis", "repeat", "flip"]].drop_duplicates().itertuples(index=False, name=None))
    print(f"Resuming: {len(done)} trials already saved.")

# for each axis, report at the given report_layers
conditions = [(l, a) for l in REPORT_LAYERS for a in AXES]
if RUN_SHUFFLED_CONTROL:
    for l in REPORT_LAYERS:
        conditions.append((l, "SHUFFLED"))
n_total = len(conditions) * N_REPEATS * 2
timing_shown = False

with tqdm(total=n_total, desc="Reporting trials") as bar:
    for rep in range(N_REPEATS):
        # randomly sample examples sentences from test set
        idx = np.random.default_rng(SEED + rep).choice(len(test_sents), N_EXAMPLES, replace=False)
        sents = [test_sents[i] for i in idx]
        for layer, axis in conditions:
            if axis == "SHUFFLED":
                # Permute the logistic regression labels
                true = np.random.default_rng(10_000 + SEED + rep).permutation(labels[layer]["LR"][idx])
            else:
                # Get the true axis label based on the current saved
                true = labels[layer][axis][idx]
            for flip in (False, True):
                bar.update(1)
                if (layer, axis, rep, flip) in done:
                    continue
                # Swap the labels
                shown = 1 - true if flip else true
                t0 = time.time()
                try:
                    p1, n_tok = report_trial(sents, shown)
                except torch.cuda.OutOfMemoryError as err:
                    raise RuntimeError("GPU out of memory. Lower N_EXAMPLES and re-run this cell.") from err
                if not timing_shown:
                    bar.write(f"Each prompt is ~{n_tok:,} tokens; one trial took {time.time() - t0:.1f} s.")
                    timing_shown = True
                # Position is for sentence within the prompt (how many has the model seen so far)
                pd.DataFrame({"layer": layer, "axis": axis, "repeat": rep, "flip": flip,
                              "position": np.arange(N_EXAMPLES), "label": shown, "p1": p1}
                             ).to_csv(results_file, mode="a", header=not os.path.exists(results_file), index=False)
print("Results saved to", results_file)

## Reporting task results


### Single axis reporting task

We check the accuracy and cross entropy after looking at the axis-based labels (ground truth) and the model's reported labels.

How do accuracy and cross-entropy change as a function of example sentence and scores provided in the prompt?

Initialize plotting settings and helper functions

In [ ]:
import matplotlib.pyplot as plt

# Initialize colors for plots
INK, INK_2, MUTED, GRID, BASELINE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
LR_COLOR = "#eb6834"
PC_RAMP = ["#0d366b", "#1c5cab", "#2a78d6", "#5598e7", "#86b6ef"]   # darker = higher-variance PC
BIN = 10
pcs_run = [a for a in AXES if a.startswith("PC")]


def style(axis):
    if axis == "LR":
        return dict(color=LR_COLOR, label="LR (morality)", ls="-")
    if axis == "SHUFFLED":
        return dict(color=MUTED, label="Shuffled labels (control)", ls="--")
    i = pcs_run.index(axis)
    step = round(i * (len(PC_RAMP) - 1) / max(len(pcs_run) - 1, 1))
    return dict(color=PC_RAMP[step], label=axis, ls="-")

def curve(axis, metric):
    # One row per trial (layer, repeat, flip); one column per bin of positions.
    units = df[df.axis == axis].pivot_table(index=["layer", "repeat", "flip"], columns="bin", values=metric)
    m, se = units.mean(axis=0).to_numpy(), (units.std(axis=0, ddof=1) / np.sqrt(len(units))).to_numpy()
    return units.columns.to_numpy() * BIN + (BIN - 1) / 2, m, se



In [ ]:
df = pd.read_csv(results_file)
df = df[df.layer.isin(REPORT_LAYERS) & (df.position < N_EXAMPLES)]
df["correct"] = ((df.p1 > 0.5).astype(int) == df.label).astype(float)
df["ce"] = -np.log(np.where(df.label == 1, df.p1, 1 - df.p1).clip(1e-6, 1)) # cross entropy
df["bin"] = df.position // BIN # Bin sentences of labeled examples

Plot accuracy and entropy curves for each layer we're looking at.

In [ ]:
order = AXES + ["SHUFFLED"]

plt.rcParams.update({"font.size": 10, "text.color": INK, "axes.labelcolor": INK_2,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.edgecolor": BASELINE})

def curve_layer(axis, metric, df_subset):
    units = df_subset[df_subset.axis == axis].pivot_table(index=["repeat", "flip"], columns="bin", values=metric)
    m, se = units.mean(axis=0).to_numpy(), (units.std(axis=0, ddof=1) / np.sqrt(len(units))).to_numpy()
    return units.columns.to_numpy() * BIN + (BIN - 1) / 2, m, se

n_rows = len(REPORT_LAYERS) + 1  # +1 for averaged row
fig, axes = plt.subplots(n_rows, 2, figsize=(11, 3.8 * n_rows), dpi=120)
if n_rows == 1:
    axes = [axes]

row_specs = [(layer, f"Layer {layer}", df[df.layer == layer]) for layer in REPORT_LAYERS]
row_specs.append((None, "All layers (averaged)", df))

for row, (layer, row_title, df_subset) in enumerate(row_specs):
    ax_acc, ax_ce = axes[row]
    for ax, metric, chance, title in [(ax_acc, "correct", 0.5, "Reporting accuracy (chance = 0.5)"),
                                      (ax_ce, "ce", np.log(2), "Cross-entropy (chance = 0.69; lower is better)")]:
        ax.axhline(chance, color=BASELINE, lw=0.8)
        for a in order:
            x, m, se = curve_layer(a, metric, df_subset)
            st = style(a)
            ax.plot(x, m, color=st["color"], ls=st["ls"], lw=1.6, label=st["label"])
            ax.fill_between(x, m - se, m + se, color=st["color"], alpha=0.10, lw=0)
        ax.set_title(f"{row_title} — {title}", loc="left", color=INK, fontsize=11)
        ax.set_xlabel("Labeled examples already in the prompt")
        ax.grid(axis="y", color=GRID, lw=0.6)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_xlim(0, N_EXAMPLES)
    ax_acc.set_ylim(0.4, 1.0)
    ax_ce.set_ylim(0, 1.2)
    ax_ce.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0))

fig.suptitle(f"{MODEL_NAME.split('/')[-1]}: can the model report its own activations?",
             x=0.01, ha="left", color=INK, fontsize=11, y=1.02)
fig.tight_layout()
fig.savefig(f"{SAVE_DIR}/reporting_curves.png", bbox_inches="tight")
plt.show()


## Control Task